In [12]:
import os
import torch
import librosa
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration, WhisperFeatureExtractor, WhisperTokenizer
from IPython.display import Audio, display
import random

In [7]:
segments_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/segment"
output_csv = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/segment_csv/results.csv"
model_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/results/finalModelV2/checkpoint-1200"

In [3]:
tokenizer = WhisperTokenizer.from_pretrained(model_dir)
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-base")
processor = WhisperProcessor(feature_extractor, tokenizer)
model = WhisperForConditionalGeneration.from_pretrained(model_dir)
model.eval()

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 512)
      (layers): ModuleList(
        (0-5): 6 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=False)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (fin

In [8]:
def add_noise(wav, snr_db):
    wav = wav / (wav.abs().max() + 1e-8)
    noise = torch.randn_like(wav)

    signal_power = wav.pow(2).mean()
    noise_power = noise.pow(2).mean()

    snr = 10 ** (snr_db / 10)
    scale = torch.sqrt(signal_power / (snr * noise_power))

    return wav + scale * noise

In [14]:
def transcribe(wav):
    inputs = processor(wav, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]


In [ ]:
files = [f for f in os.listdir(segments_dir) if f.endswith(".wav")]
files = random.sample(files, 10)

for file in files:
    path = os.path.join(segments_dir, file)

    wav, sr = librosa.load(path, sr=16000)
    wav = torch.tensor(wav)

    print(f"----------------------{file}-----------------------------")

    clean_audio = wav
    clean_text = transcribe(clean_audio)

    print("CLEAN:")
    display(Audio(clean_audio.numpy(), rate=sr))
    print(clean_text)

    for snr in [5, 10, 15]:
        noisy_audio = add_noise(wav, snr)
        noisy_text = transcribe(noisy_audio)

        print(f"\nSNR {snr}:")
        display(Audio(noisy_audio.numpy(), rate=sr))
        print(noisy_text)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


----------------------segment_7406.wav-----------------------------


`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.ge

CLEAN:


Yan yew kitabê cıyo roşınkerdov bıbo.

SNR 5:


...

Tı do zahmet li dîne.
----------------------segment_24776.wav-----------------------------
CLEAN:


Kam îman bûhomay bîyar o

SNR 5:


Kan iman bihomay dîyare o

SNR 10:


Kan iman bûhomay bîyar o

SNR 15:


Kan iman bûhomay bîyar o
----------------------segment_7152.wav-----------------------------
